# Тест качества собранных пазлов

Оценивает конфиг банка терминов (`configs/*.json`) и реальный генератор
(`js/puzzle-generator.js`) по 5 метрикам:

| # | Метрика | Что значит | Цель |
|---|---------|------------|------|
| 1 | Пазл собрался + воспроизводимость | генератор не падает, доска=16, каждый термин в ≤1 категории, два прогона с одним сидом идентичны, хэш совпадает с эталоном | всё True |
| 2 | Доля и кол-во широких категорий (≥20 терминов) | широкие = размытые, плохо | 0 |
| 3 | Средняя глубина категории | среднее число терминов на категорию | контроль раздувания |
| 4 | Доля терминов в ≥1 широкой категории | сколько банка утонуло в размытом | 0 |
| 5 | Термин хоть сколько-то распространён | LLM-as-judge + web search по источникам, двойная проверка + калибровка | высокая доля |

Вся логика — в `tests/puzzle_eval.py` (одна точка правды). Ноутбук только запускает.

In [1]:
import sys, glob
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'tests'))
import puzzle_eval as pe

CONFIGS = sorted(glob.glob('configs/*.json'))
CONFIGS

['configs/Evals-1-multilang.json',
 'configs/category-templates-new.json',
 'configs/category-templates.json',
 'configs/evals-2.json',
 'configs/evals-3.json',
 'configs/evals-4.json']

## Метрики 1–4 по всем конфигам (детерминированы, ключ не нужен)

In [2]:
rows = []
for cfg in CONFIGS:
    terms = pe.load_terms(cfg, lang='en')
    st = pe.structural_metrics(terms)
    try:
        asm = pe.assembly_metrics(cfg)
        m1 = all(a.ok for a in asm)
    except Exception as e:
        m1 = f'ERR: {e}'
    rows.append({
        'config': Path(cfg).name,
        'M1_ok': m1,
        'terms': st.n_terms, 'cats': st.n_tags,
        'M2_broad': st.broad_count, 'M2_broad_share': round(st.broad_share, 2),
        'M3_avg_depth': round(st.avg_category_depth, 2),
        'M4_in_broad_share': round(st.terms_in_broad_share, 2),
    })

hdr = ['config','M1_ok','terms','cats','M2_broad','M2_broad_share','M3_avg_depth','M4_in_broad_share']
w = {h: max(len(h), *(len(str(r[h])) for r in rows)) for h in hdr}
print('  '.join(h.ljust(w[h]) for h in hdr))
for r in rows:
    print('  '.join(str(r[h]).ljust(w[h]) for h in hdr))

config                       M1_ok  terms  cats  M2_broad  M2_broad_share  M3_avg_depth  M4_in_broad_share
Evals-1-multilang.json       True   16     4     0         0.0             4             0.0              
category-templates-new.json  True   40     10    0         0.0             4             0.0              
category-templates.json      False  40     10    0         0.0             4             0.0              
evals-2.json                 True   16     4     0         0.0             4             0.0              
evals-3.json                 True   16     4     0         0.0             4             0.0              
evals-4.json                 True   16     4     0         0.0             4             0.0              


## Подробный отчёт по одному конфигу

In [3]:
report = pe.run_report('configs/category-templates-new.json', lang='en', run_metric5=False)
# распределение размеров категорий
report['structural'].category_sizes

=== category-templates-new.json  (lang=en) ===

[M1] Сборка + воспроизводимость:
  normal   : собрался=True воспроизводим=True доска16=True 1термин-1кат=True | hash=8bdc107d0187c3b9 эталон ✓
  advanced : собрался=True воспроизводим=True доска16=True 1термин-1кат=True | hash=425eebd1c03ed898 эталон ✓

[M2–M4] Структура банка:
  терминов=40, категорий=10 (генерируемых≥4: 10)
    M2 широкие категории (≥20): 0 шт (0%) -> —
    M3 средняя глубина категории: 4.00 (медиана 4.0)
    M4 доля терминов в широкой категории: 0% (0 шт)

[M5] Распространённость терминов:
  пропущено (run_metric5=False)


{'Technical Failure Modes': 4,
 'AI Governance Mechanisms': 4,
 'Philosophical Concepts': 4,
 'Societal Risks': 4,
 'Alignment Research Areas': 4,
 'Thought Experiments': 4,
 'Technical Safety Approaches': 4,
 'Systemic Risks': 4,
 'AI Development Principles': 4,
 'Outer Alignment Concepts': 4}

## Метрика 5 — распространённость терминов (LLM + web search)

Требует `ANTHROPIC_API_KEY` и `pip install anthropic`.

**Двойная проверка** заложена в дизайн:
1. На каждый термин — два независимых прохода: `researcher` (ищет подтверждение) и `skeptic` (пытается опровергнуть). `recognized=True` только если оба согласны; расхождение → `needs_human`.
2. Перед доверием метрике — **калибровка** на заведомо реальных и заведомо выдуманных терминах. Доверяем только если `real_recall` и `fake_rejection` близки к 1.0. Это и есть «дважды проверить, что LLM найдёт и насудит».

In [4]:
# 1) Сначала калибруем судью
try:
    calib = pe.calibrate_judge(model='claude-sonnet-4-6')
    print(f"real_recall={calib['real_recall']:.0%}  fake_rejection={calib['fake_rejection']:.0%}  "
          f"trustworthy={calib['trustworthy']}")
except RuntimeError as e:
    print('калибровка пропущена:', e)

калибровка пропущена: нет ANTHROPIC_API_KEY — метрика 5 пропущена


In [5]:
# 2) Только если калибровка прошла — судим реальные термины конфига
try:
    terms = pe.load_terms('configs/category-templates-new.json', lang='en')
    m5 = pe.recognizability_metrics(terms, model='claude-sonnet-4-6')
    print(f"опознано: {m5['recognized_share']:.0%}")
    print('подозрительные:', m5['suspicious_terms'] or '—')
    print('на ручную проверку:', m5['needs_human_review'] or '—')
except RuntimeError as e:
    print('метрика 5 пропущена:', e)

метрика 5 пропущена: нет ANTHROPIC_API_KEY — метрика 5 пропущена


## Что важно знать про метрику 1 (воспроизводимость)

Да, воспроизводимость нужна — иначе препод не гарантирует одинаковый тест.
Но в текущем коде есть две оговорки, которые тест ловит/подсвечивает:

1. **Кросс-браузерность.** Перетасовка сделана через `arr.sort(() => rand() - 0.5)`. Это детерминировано в пределах одного движка, но V8 (Chrome/Edge/Node), SpiderMonkey (Firefox) и JSC (Safari) сортируют разными алгоритмами → один и тот же сид может дать **разный пазл в разных браузерах**. Эталон (`golden.json`) фиксирует именно V8-результат. Фикс: заменить три `sort(()=>rand()-0.5)` на сидированный Fisher–Yates.
2. **Ловушки в advanced.** Decoy-плитки берутся не из переданного конфига, а из глобального `CONCEPT_DEFINITIONS`, который грузится отдельно из `category-templates-new.json`. Для кастомных комнат это значит, что ловушки протекают из дефолтного банка. Тест подставляет defs из конфига, чтобы доска заполнялась до 16.

`golden.json` создаётся при первом прогоне. Пересоздать эталон после намеренного изменения конфига: `pe.assembly_metrics(cfg, update_golden=True)`.